# Hearing Care Partner VoC — survey, synthetic data and analysis

---



## Part 1: Generate synthetic data

In [2]:
import pandas as pd
import numpy as np

In [3]:
import numpy as np

rng = np.random.default_rng(42)   # seed = same "random" data every run
n = 300

clinics = pd.DataFrame({
    "ClinicID": [f"CL{str(i).zfill(4)}" for i in range(1, n + 1)],
    "Region": rng.choice(["Northeast", "Southeast", "Midwest", "West"], n),
    "ClinicSize": rng.choice(["Small", "Medium", "Large"], n, p=[0.5, 0.35, 0.15]),
    "AccountTier": rng.choice(["Bronze", "Silver", "Gold"], n, p=[0.4, 0.4, 0.2]),
})

# Operational data (O data)
clinics["avg_repair_days"] = rng.normal(10, 4, n).clip(3, 25).round(1)
clinics.loc[clinics["Region"] == "West", "avg_repair_days"] += 3   # planted pattern

clinics["order_delay_rate"] = rng.beta(2, 15, n).round(3)          # share of orders delayed
clinics["support_tickets_90d"] = rng.poisson(4, n)

clinics.head()

,ClinicID,Region,ClinicSize,AccountTier,avg_repair_days,order_delay_rate,support_tickets_90d
0,CL0001,Northeast,Medium,Bronze,9.6,0.116,6
1,CL0002,West,Large,Gold,17.5,0.009,2
2,CL0003,Midwest,Small,Silver,3.0,0.225,2
3,CL0004,Southeast,Medium,Bronze,4.0,0.054,6
4,CL0005,Southeast,Small,Bronze,6.3,0.206,5


In [4]:
# 60% of clinics respond
resp = clinics.sample(frac=0.6, random_state=42).reset_index(drop=True)
m = len(resp)

# Hidden satisfaction score, driven by operations + random noise
latent = (8.5
          - 0.30 * (resp["avg_repair_days"] - 10)      # slow repairs hurt most
          - 8.0  * (resp["order_delay_rate"] - 0.12)   # delayed orders hurt
          - 0.15 * (resp["support_tickets_90d"] - 4)   # many tickets = problems
          + rng.normal(0, 1.5, m))                     # people are unpredictable

resp["NPS"] = latent.round().clip(0, 10).astype(int)

# CSAT follows the same score, mapped to 5 labels
csat_labels = ["Extremely dissatisfied", "Dissatisfied", "Neutral",
               "Satisfied", "Extremely satisfied"]
csat_score = ((latent + rng.normal(0, 1, m)) / 2.2).round().clip(1, 5).astype(int)
resp["CSAT"] = [csat_labels[s - 1] for s in csat_score]

print(m, "responses")
print(resp["NPS"].value_counts().sort_index())

180 responses
NPS
2      1
3      3
4      5
5      8
6     13
7     34
8     37
9     30
10    49
Name: count, dtype: int64


In [5]:
promoters = (resp["NPS"] >= 9).mean()
detractors = (resp["NPS"] <= 6).mean()
print("NPS:", round((promoters - detractors) * 100, 1))

NPS: 27.2


In [6]:
touchpoints = ["Placing orders", "Repairs", "Sales rep visits",
               "Product training", "Customer support"]
use_prob = [0.9, 0.6, 0.5, 0.3, 0.45]   # how often each is used

nes_labels = ["Very difficult", "Difficult", "Somewhat difficult",
              "Neither easy nor difficult", "Somewhat easy", "Easy", "Very easy"]

# Ease drivers: specific ops metric for orders/repairs, general satisfaction for the rest
ease_base = {
    "Placing orders":   5.5 - 15 * (resp["order_delay_rate"] - 0.12),
    "Repairs":          5.5 - 0.35 * (resp["avg_repair_days"] - 10),
    "Sales rep visits": 4.0 + 0.25 * latent,
    "Product training": 4.0 + 0.2 * latent,
    "Customer support": 5.5 - 0.3 * (resp["support_tickets_90d"] - 4),
}

used = np.column_stack([rng.random(m) < p for p in use_prob])
used[used.sum(axis=1) == 0, 0] = True    # everyone used at least one thing

resp["TP_Used"] = [",".join(t for t, u in zip(touchpoints, row) if u) for row in used]

for i, tp in enumerate(touchpoints):
    score = (ease_base[tp] + rng.normal(0, 1, m)).round().clip(1, 7).astype(int)
    resp[f"NES_{i+1}"] = [nes_labels[s - 1] if used[r, i] else np.nan
                          for r, s in enumerate(score)]

resp[["ClinicID", "NPS", "TP_Used", "NES_1", "NES_2"]].head()

,ClinicID,NPS,TP_Used,NES_1,NES_2
0,CL0204,8,"Repairs,Customer support",NaN,Easy
1,CL0267,9,"Placing orders,Repairs,Sales rep visits,Produc...",Easy,Very easy
2,CL0153,7,Customer support,NaN,NaN
3,CL0010,10,"Placing orders,Repairs,Sales rep visits,Custom...",Somewhat easy,Very easy
4,CL0234,10,Placing orders,Somewhat difficult,NaN


In [7]:
detractor_text = {
    "repairs": ["Repairs take way too long, my patients keep asking where their devices are.",
                "Repair turnaround is the biggest problem for us.",
                "We waited almost three weeks for a repair to come back."],
    "orders":  ["Orders are often delayed or arrive incomplete.",
                "Shipping delays make it hard to schedule fittings.",
                "Order status is unclear and backorders are common."],
    "support": ["It takes several calls to get a support issue resolved.",
                "Support is hard to reach and slow to respond.",
                "We keep opening tickets for the same problems."],
}
passive_text = ["Faster repairs would make a big difference.",
                "Better order tracking would help.",
                "More regular training on new products.",
                "Quicker answers from support."]
promoter_text = ["Our sales rep is great and always responds quickly.",
                 "Product quality is excellent and patients love it.",
                 "Training sessions are really helpful for our staff.",
                 "Ordering is easy and reliable."]

def worst_problem(row):
    # compare each issue relative to its typical value
    scores = {"repairs": row["avg_repair_days"] / 10,
              "orders":  row["order_delay_rate"] / 0.12,
              "support": row["support_tickets_90d"] / 4}
    return max(scores, key=scores.get)

resp["Detractor_Fix"] = None
resp["Contact_OK"] = None
resp["Passive_Improve"] = None
resp["Promoter_Keep"] = None

for i, row in resp.iterrows():
    writes_comment = rng.random() < 0.7          # not everyone writes something
    if row["NPS"] <= 6:
        if writes_comment:
            resp.at[i, "Detractor_Fix"] = rng.choice(detractor_text[worst_problem(row)])
        resp.at[i, "Contact_OK"] = "Yes" if rng.random() < 0.6 else "No"
    elif row["NPS"] <= 8:
        if writes_comment:
            resp.at[i, "Passive_Improve"] = rng.choice(passive_text)
    else:
        if writes_comment:
            resp.at[i, "Promoter_Keep"] = rng.choice(promoter_text)

resp.loc[resp["NPS"] <= 6, ["ClinicID", "NPS", "avg_repair_days", "Detractor_Fix", "Contact_OK"]].head(8)

,ClinicID,NPS,avg_repair_days,Detractor_Fix,Contact_OK
10,CL0238,6,19.0,Repair turnaround is the biggest problem for us.,Yes
15,CL0222,6,13.5,None,No
20,CL0079,6,14.1,Orders are often delayed or arrive incomplete.,Yes
25,CL0043,6,13.0,None,Yes
28,CL0158,6,9.1,None,Yes
32,CL0034,3,18.2,Support is hard to reach and slow to respond.,Yes
42,CL0287,4,12.7,Order status is unclear and backorders are com...,Yes
43,CL0061,6,12.8,Repair turnaround is the biggest problem for us.,Yes


In [8]:
# Add the response metadata Qualtrics would create
resp["ResponseId"] = [f"R_SYN{str(i).zfill(4)}" for i in range(1, m + 1)]
start = pd.Timestamp("2026-07-01")
resp["RecordedDate"] = start + pd.to_timedelta(rng.integers(0, 92, m), unit="D")

survey_cols = ["ResponseId", "RecordedDate", "NPS", "CSAT", "TP_Used",
               "NES_1", "NES_2", "NES_3", "NES_4", "NES_5",
               "Detractor_Fix", "Contact_OK", "Passive_Improve", "Promoter_Keep",
               "ClinicID", "Region", "ClinicSize", "AccountTier"]

survey = resp[survey_cols]
survey.to_csv("survey_responses_synthetic.csv", index=False)
clinics.to_csv("clinic_operations_synthetic.csv", index=False)

print(survey.shape, clinics.shape)

(180, 18) (300, 7)


In [9]:
import os
print([f for f in os.listdir() if f.endswith(".csv")])

['survey_responses_synthetic.csv', 'clinic_operations_synthetic.csv']


In [10]:
from google.colab import files
files.download("survey_responses_synthetic.csv")
files.download("clinic_operations_synthetic.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Part 2: Analysis

In [11]:
import pandas as pd
import numpy as np

survey = pd.read_csv("survey_responses_synthetic.csv")
ops = pd.read_csv("clinic_operations_synthetic.csv")

# Join survey (X data) with operations (O data)
df = survey.merge(ops[["ClinicID", "avg_repair_days", "order_delay_rate", "support_tickets_90d"]],
                  on="ClinicID", how="left")

# Classify each response
df["NPS_Group"] = pd.cut(df["NPS"], bins=[-1, 6, 8, 10],
                         labels=["Detractor", "Passive", "Promoter"])

print(df.shape)
print(df["NPS_Group"].value_counts())

(180, 22)
NPS_Group
Promoter     79
Passive      71
Detractor    30
Name: count, dtype: int64


In [12]:
def nps(scores):
    return round(((scores >= 9).mean() - (scores <= 6).mean()) * 100, 1)

print("Overall NPS:", nps(df["NPS"]), "| n =", len(df))

for col in ["Region", "ClinicSize", "AccountTier"]:
    summary = df.groupby(col).agg(n=("NPS", "size"), NPS=("NPS", nps)).sort_values("NPS")
    print(f"\n{col}")
    print(summary)

Overall NPS: 27.2 | n = 180

Region
            n   NPS
Region             
West       36  11.1
Southeast  55  23.6
Midwest    47  29.8
Northeast  42  42.9

ClinicSize
             n   NPS
ClinicSize          
Small       89  23.6
Medium      63  28.6
Large       28  35.7

AccountTier
              n   NPS
AccountTier          
Gold         36  13.9
Bronze       73  27.4
Silver       71  33.8


In [13]:
drivers = ["avg_repair_days", "order_delay_rate", "support_tickets_90d"]

print(df.groupby("NPS_Group", observed=True)[drivers].mean().round(2))

           avg_repair_days  order_delay_rate  support_tickets_90d
NPS_Group                                                        
Detractor            13.94              0.17                 5.33
Passive              11.82              0.12                 3.93
Promoter              8.88              0.10                 3.71


In [14]:
print(df[drivers + ["NPS"]].corr()["NPS"].drop("NPS").sort_values().round(2))

avg_repair_days       -0.52
order_delay_rate      -0.31
support_tickets_90d   -0.25
Name: NPS, dtype: float64


In [15]:
nes_map = {"Very difficult": 1, "Difficult": 2, "Somewhat difficult": 3,
           "Neither easy nor difficult": 4, "Somewhat easy": 5, "Easy": 6, "Very easy": 7}

touchpoints = ["Placing orders", "Repairs", "Sales rep visits",
               "Product training", "Customer support"]

rows = []
for i, tp in enumerate(touchpoints, start=1):
    scores = df[f"NES_{i}"].map(nes_map).dropna()
    rows.append({"Touchpoint": tp,
                 "n": len(scores),
                 "Avg_ease": round(scores.mean(), 2),
                 "Pct_easy": round((scores >= 5).mean() * 100, 1),
                 "Pct_difficult": round((scores <= 3).mean() * 100, 1)})

nes_summary = pd.DataFrame(rows).sort_values("Avg_ease")
print(nes_summary)

         Touchpoint    n  Avg_ease  Pct_easy  Pct_difficult
1           Repairs  121      5.02      65.3           14.9
0    Placing orders  163      5.39      76.7           10.4
3  Product training   56      5.50      85.7            7.1
4  Customer support   89      5.52      83.1            2.2
2  Sales rep visits  101      6.01      96.0            0.0


In [16]:
themes = {
    "Repairs":  ["repair", "turnaround", "devices"],
    "Orders":   ["order", "shipping", "backorder", "tracking"],
    "Support":  ["support", "ticket", "calls"],
    "Sales rep": ["rep"],
    "Training": ["training"],
    "Product":  ["product quality", "patients love"],
}

def tag(text):
    if pd.isna(text):
        return None
    text = text.lower()
    for theme, words in themes.items():
        if any(w in text for w in words):
            return theme
    return "Other"

for col in ["Detractor_Fix", "Passive_Improve", "Promoter_Keep"]:
    counts = df[col].apply(tag).value_counts()
    print(f"\n{col} (n = {counts.sum()})")
    print(counts)


Detractor_Fix (n = 22)
Detractor_Fix
Repairs    10
Orders      7
Support     5
Name: count, dtype: int64

Passive_Improve (n = 54)
Passive_Improve
Repairs     17
Support     15
Training    14
Orders       8
Name: count, dtype: int64

Promoter_Keep (n = 54)
Promoter_Keep
Product      19
Orders       13
Training     13
Sales rep     9
Name: count, dtype: int64


In [17]:
at_risk = df[(df["NPS_Group"] == "Detractor") & (df["Contact_OK"] == "Yes")].copy()

tier_rank = {"Gold": 1, "Silver": 2, "Bronze": 3}
at_risk["tier_rank"] = at_risk["AccountTier"].map(tier_rank)

at_risk = at_risk.sort_values(["tier_rank", "NPS"])[
    ["ClinicID", "Region", "AccountTier", "NPS", "avg_repair_days",
     "order_delay_rate", "Detractor_Fix"]]

print(len(at_risk), "clinics to contact")
at_risk.head(10)

25 clinics to contact


,ClinicID,Region,AccountTier,NPS,avg_repair_days,order_delay_rate,Detractor_Fix
57,CL0093,West,Gold,4,14.0,0.210,NaN
131,CL0113,Midwest,Gold,5,5.5,0.320,Shipping delays make it hard to schedule fitti...
174,CL0090,West,Gold,5,13.9,0.414,Order status is unclear and backorders are com...
10,CL0238,Southeast,Gold,6,19.0,0.071,Repair turnaround is the biggest problem for us.
119,CL0207,West,Gold,6,12.3,0.181,Order status is unclear and backorders are com...
177,CL0027,Southeast,Silver,3,17.2,0.161,We keep opening tickets for the same problems.
141,CL0144,Southeast,Silver,4,10.3,0.267,NaN
65,CL0031,Southeast,Silver,5,10.0,0.057,It takes several calls to get a support issue ...
67,CL0257,Midwest,Silver,5,14.4,0.082,We waited almost three weeks for a repair to c...
138,CL0179,West,Silver,5,14.9,0.079,We waited almost three weeks for a repair to c...


In [18]:
ops["Responded"] = ops["ClinicID"].isin(df["ClinicID"])
print(ops.groupby("Responded")[["avg_repair_days", "order_delay_rate", "support_tickets_90d"]].mean().round(2))

           avg_repair_days  order_delay_rate  support_tickets_90d
Responded                                                        
False                10.66              0.13                 4.03
True                 10.88              0.12                 4.07


In [19]:
df.to_csv("vox_joined.csv", index=False)
at_risk.to_csv("at_risk_accounts.csv", index=False)
nes_summary.to_csv("nes_summary.csv", index=False)